## Bước 1: Cài đặt thư viện

In [2]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet
!pip install --no-deps "xformers<0.0.29" peft accelerate bitsandbytes --quiet
!pip install datasets transformers scikit-learn pandas PyYAML --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 421.9/421.9 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 101.0 MB/s

## Bước 2: Clone repo từ GitHub

In [3]:
!git clone https://github.com/ba0-123/banking-intent-unsloth.git
%cd banking-intent-unsloth
!ls -la

Cloning into 'banking-intent-unsloth'...
remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 15 (delta 3), reused 15 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (15/15), 13.83 KiB | 13.83 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/banking-intent-unsloth
total 36
drwxr-xr-x 5 root root 4096 Apr 27 13:52 .
drwxr-xr-x 1 root root 4096 Apr 27 13:52 ..
-rw-r--r-- 1 root root 7143 Apr 27 13:52 banking77_colab.ipynb
drwxr-xr-x 2 root root 4096 Apr 27 13:52 configs
drwxr-xr-x 8 root root 4096 Apr 27 13:52 .git
-rw-r--r-- 1 root root 4678 Apr 27 13:52 README.md
drwxr-xr-x 2 root root 4096 Apr 27 13:52 scripts


## Bước 3: Tiền xử lý dữ liệu BANKING77

In [4]:
!python scripts/preprocess_data.py --config configs/train.yaml

📥 Đang tải BANKING77 từ CSV...
✅ Tổng số mẫu: 13083
✅ Số intents: 77
💾 Saved:
   train.csv : 11120 samples
   test.csv  : 1963 samples
   label_map.json : 77 intents


In [5]:
# Kiểm tra dữ liệu
import pandas as pd, json

df_train = pd.read_csv('sample_data/train.csv')
df_test  = pd.read_csv('sample_data/test.csv')

print(f'Train: {len(df_train)} mẫu')
print(f'Test : {len(df_test)} mẫu')
print(f'\nPhân phối intents (top 5):')
print(df_train['intent'].value_counts().head())
print(f'\nVí dụ:')
print(df_train[['text', 'intent']].head(3).to_string())

Train: 11120 mẫu
Test : 1963 mẫu

Phân phối intents (top 5):
intent
card_payment_fee_charged                            193
direct_debit_payment_not_recognised                 189
balance_not_updated_after_cheque_or_cash_deposit    188
wrong_amount_of_cash_received                       187
cash_withdrawal_charge                              184
Name: count, dtype: int64

Ví dụ:
                                                                                                                          text                                   intent
0                                                                                        help! i need to cancel a transaction.                          cancel_transfer
1  i have to transfer money over to my mother, and i have tried numerous times and all that keeps happening is error messages.                  beneficiary_not_allowed
2                                                                                                      my transfer

## Bước 4: Fine-tuning với Unsloth

In [15]:
!python scripts/train.py --config configs/train.yaml

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📌 Số intents: 77

🚀 Tải model: unsloth/Llama-3.2-1B-Instruct
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 146/146 [00:00<00:00, 156.95it/s]
Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.8 patched 16 layers with

## Bước 5: Backup Checkpoint lên Drive

In [21]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/banking77-checkpoint'
os.makedirs(DRIVE_DIR, exist_ok=True)

# Copy checkpoint
src = 'outputs/llama32-banking77/checkpoint-final'
shutil.copytree(src, DRIVE_DIR + '/checkpoint-final', dirs_exist_ok=True)
shutil.copytree('sample_data', DRIVE_DIR + '/sample_data', dirs_exist_ok=True)

print(f'Đã backup lên: {DRIVE_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã backup lên: /content/drive/MyDrive/banking77-checkpoint


## Bước 6: Inference — Demo kết quả

In [17]:
import sys
sys.path.insert(0, '.')
from scripts.inference import IntentClassification

# Tải model
classifier = IntentClassification('configs/inference.yaml')

[IntentClassification] Đang tải model từ: outputs/llama32-banking77/checkpoint-final
==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


<string>:42: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-1b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


[IntentClassification] ✅ Model đã sẵn sàng!


In [18]:
# ── Demo với nhiều ví dụ ──
test_messages = [
    'I lost my credit card',
    'What is my current balance?',
    'I want to transfer money to another account',
    'My card payment was declined',
    'How do I change my PIN?',
    'I was charged twice for the same transaction',
    'What is the exchange rate for USD to EUR?',
    'I want to cancel my card',
    'When will my refund arrive?',
    'How do I activate my new card?',
]

print('=' * 60)
print('  BANKING INTENT CLASSIFICATION — DEMO')
print('=' * 60)
for msg in test_messages:
    label = classifier(msg)
    print(f'\nInput  : {msg}')
    print(f'Intent : {label}')
print('=' * 60)

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  BANKING INTENT CLASSIFICATION — DEMO


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input  : I lost my credit card
Intent : lost_or_stolen_card


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...


Input  : I was charged twice for the same transaction
Intent : transaction_charged_twice

Input  : What is the exchange rate for USD to EUR?
Intent : exchange_rate


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input  : I want to cancel my card
Intent : terminate_account


Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Input  : When will my refund arrive?
Intent : Refund_not_showing_up

Input  : How do I activate my new card?
Intent : activate_my_card


In [29]:
your_message = 'Why did I get charged a fee?'

label = classifier(your_message)
print(f'Message: {your_message}')
print(f'Intent : {label}')

Both `max_new_tokens` (=20) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Message: Why did I get charged a fee?
Intent : card_payment_fee_charged


## Bước 7: Xem kết quả đánh giá

In [31]:
# Xem accuracy và classification report
with open('outputs/llama32-banking77/eval_results.txt') as f:
    print(f.read())

Test Accuracy: 0.9149

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.97      0.93      0.95        30
                                activate_my_card       1.00      0.87      0.93        30
                                       age_limit       1.00      1.00      1.00        23
                         apple_pay_or_google_pay       1.00      1.00      1.00        25
                                     atm_support       0.95      0.95      0.95        19
                                automatic_top_up       1.00      0.92      0.96        25
         balance_not_updated_after_bank_transfer       0.87      0.84      0.86        32
balance_not_updated_after_cheque_or_cash_deposit       0.94      0.88      0.91        33
                         beneficiary_not_allowed       0.96      0.86      0.91        29
                                 cancel_transfer       0.94      1.00      0